# 04.06 Document Extraction Validation

**Phase 04 — Document Processing & OCR**  
**Objective:** Validate OCR extraction outputs from Notebook 04 and classification/linkage contracts from Notebook 05 using confidence thresholds, required-field checks, cross-document consistency rules, and explicit human-review routing. Low-confidence extractions are **never** automatically marked as verified.


### 1. Environment Setup & Validation Policy


In [ ]:
import os
import sys
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

def _find_root(p: Path = Path.cwd()) -> Path:
    for parent in [p.resolve(), *p.resolve().parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths
paths = DocumentProcessingPaths(PROJECT_ROOT)

OCR_ENTITIES_JSON = paths.interim_doc_proc / 'extracted_ocr' / 'ocr_extracted_entities.json'
CLASSIFICATION_JSON = paths.interim_doc_proc / 'document_classification_linkage.json'
VALIDATION_DIR = paths.processed_dir / 'validation'
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

VAL_REPORT_JSON = VALIDATION_DIR / 'document_extraction_validation_report.json'
VAL_TABLE_CSV = VALIDATION_DIR / 'document_pipeline_validation_report.csv'

CONFIDENCE_POLICY = {
    'verified_min_conf': 0.85,
    'review_min_conf': 0.60,
    'min_text_length': 10,
}

print(f"Project Root        : {paths.root}")
print(f"OCR Entities JSON   : {OCR_ENTITIES_JSON} (exists={OCR_ENTITIES_JSON.exists()})")
print(f"Classification JSON : {CLASSIFICATION_JSON} (exists={CLASSIFICATION_JSON.exists()})")
print(f"Validation Target   : {VAL_REPORT_JSON}")


### 2. Confidence-Gated Field Validation & Human-Review Routing


In [ ]:
ocr_data = json.loads(OCR_ENTITIES_JSON.read_text(encoding='utf-8')) if OCR_ENTITIES_JSON.exists() else {}
clf_data = json.loads(CLASSIFICATION_JSON.read_text(encoding='utf-8')) if CLASSIFICATION_JSON.exists() else {}
clf_map = {d['document_id']: d for d in clf_data.get('documents', [])}

doc_validation_rows = []
for doc_id, rec in ocr_data.items():
    avg_conf = float(rec.get('average_confidence', 0.0))
    txt_len = int(rec.get('full_text_length', 0))
    clf_info = clf_map.get(doc_id, {})
    pred_cat = clf_info.get('predicted_category', 'unclassified')

    issues = []
    if txt_len < CONFIDENCE_POLICY['min_text_length']:
        issues.append('INCOMPLETE_OR_EMPTY_OCR_TEXT')
    if pred_cat == 'certificate_of_indigency' and not rec.get('has_indigency_cues', False):
        issues.append('MISSING_INDIGENCY_KEYWORDS')
    if pred_cat in {'electricity_bill', 'water_bill', 'expense_quotation'} and not rec.get('detected_amounts'):
        issues.append('MISSING_REQUIRED_MONETARY_AMOUNT')
    if not rec.get('detected_dates'):
        issues.append('MISSING_DOCUMENT_DATE')

    if avg_conf >= CONFIDENCE_POLICY['verified_min_conf'] and not issues:
        val_status = 'VERIFIED'
        human_review = False
    elif avg_conf >= CONFIDENCE_POLICY['review_min_conf']:
        val_status = 'NEEDS_HUMAN_REVIEW'
        human_review = True
    else:
        val_status = 'LOW_CONFIDENCE_REJECTED'
        human_review = True

    doc_validation_rows.append({
        'document_id': doc_id,
        'file_name': rec.get('file_name', doc_id),
        'predicted_category': pred_cat,
        'ocr_avg_confidence': round(avg_conf, 4),
        'text_length': txt_len,
        'validation_status': val_status,
        'human_review_required': human_review,
        'issues': '; '.join(issues) if issues else 'NONE',
    })

df_doc_val = pd.DataFrame(doc_validation_rows)
print(f"Validated {len(df_doc_val):,} extracted documents.")
if not df_doc_val.empty:
    display(df_doc_val['validation_status'].value_counts().rename_axis('Status').reset_index(name='Count'))
    display(df_doc_val.head(15))


### 3. End-to-End Pipeline Integrity Checks & JSON Export


In [ ]:
checks = [
    ('DOC-VAL-01', 'Document inventory manifest present', paths.doc_inventory_manifest.exists()),
    ('DOC-VAL-02', 'OCR extracted entities JSON present', OCR_ENTITIES_JSON.exists()),
    ('DOC-VAL-03', 'Document classification & linkage JSON present', CLASSIFICATION_JSON.exists()),
    ('DOC-VAL-04', 'Held-out test split isolated (test_manifest_v2.csv)', (paths.splits_dir / 'test_manifest_v2.csv').exists()),
    ('DOC-VAL-05', 'Low-confidence extractions routed to human review', True if df_doc_val.empty else bool((df_doc_val.loc[df_doc_val['ocr_avg_confidence'] < CONFIDENCE_POLICY['review_min_conf'], 'human_review_required']).all())),
]
df_checks = pd.DataFrame([{'check_id': cid, 'description': d, 'status': 'PASS' if ok else 'WARNING'} for cid, d, ok in checks])
df_checks.to_csv(VAL_TABLE_CSV, index=False, encoding='utf-8')

report_payload = {
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'confidence_policy': CONFIDENCE_POLICY,
    'total_documents_validated': int(len(df_doc_val)),
    'status_counts': df_doc_val['validation_status'].value_counts().to_dict() if not df_doc_val.empty else {},
    'human_review_required_count': int(df_doc_val['human_review_required'].sum()) if not df_doc_val.empty else 0,
    'pipeline_checks': df_checks.to_dict('records'),
    'document_validations': df_doc_val.to_dict('records'),
}
with open(VAL_REPORT_JSON, 'w', encoding='utf-8') as f:
    json.dump(report_payload, f, indent=2)

display(df_checks)
print(f"[OK] Saved Phase 04 extraction validation report -> {VAL_REPORT_JSON}")
